"""
# STEP 0: Simple API Test

This cell performs a quick test to ensure the OpenAI API is working correctly in this environment.

**What it does:**

1. Loads environment variables from the `.env` file to get the OpenAI API key.
2. Assigns the API key to the OpenAI client.
3. Defines a simple helper function `call_openai` to send prompts to the model.
4. Sends a test prompt `"Say hello."` to the model.
5. Prints the model's response to verify everything is working.

This is a minimal test to confirm that:
- The API key is correctly loaded.
- The new OpenAI Python SDK syntax (`openai.chat.completions.create`) works.
- The model can respond to a basic prompt.
"""


In [ ]:
# Single-cell test: Load key and ask OpenAI to say hello

import os
from dotenv import load_dotenv
import openai

# Load environment variables
load_dotenv()

# Assign API key
openai.api_key = os.getenv("OPENAI_API_KEY")
print("API Key loaded:", openai.api_key.startswith("sk-"))  # True if key is valid

# Simple helper function for chat
def call_openai(prompt, model="gpt-4.1-nano", temperature=0):
    response = openai.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=temperature
    )
    return response.choices[0].message.content.strip()

# Test prompt
test_prompt = "Say hello."
response = call_openai(test_prompt)
print("API Test Result:")
print(response)


# STEP 1: Setup

## 1.1 Sentiment Analysis
- Prompt
- Function

In [ ]:
## ==== TODO: MODIFY PROMPT ===

# --- Sentiment Prompt v1 (Zero-Shot) ---

sentiment_prompt_v1 = """
Classify this customer message: "I ordered the wireless mouse last week and it arrived earlier than expected. 
The packaging was secure and the setup was incredibly simple. 
I've been using it daily for work and the battery life is impressive — I haven't had to recharge it yet. 
The tracking is smooth, and it feels comfortable even after long hours. 
For the price, I honestly think it's an excellent value."
"""


In [ ]:
from collections import Counter

def run_classification_test(prompt, runs=15, log_file=None, label="Sentiment Test"):
    print(f"\nRunning {label} - {runs} iterations...\n")
    
    results = []
    
    for i in range(runs):
        result = call_openai(prompt).strip()
        results.append(result)
        print(f"Run {i+1}: {result}")
    
    counter = Counter(results)
    most_common_response, count = counter.most_common(1)[0]
    consistency = (count / runs) * 100
    
    print("\nMost Common Response:", most_common_response)
    print("Consistency:", f"{consistency:.2f}%")
    
    if log_file:
        with open(log_file, "a", encoding="utf-8") as f:
            f.write(f"\n--- {label} ({runs} runs) ---\n")
            for i, res in enumerate(results, 1):
                f.write(f"Run {i}: {res}\n")
            f.write(f"Consistency: {consistency:.2f}%\n")
            f.write("=" * 60 + "\n")
    
    return consistency


## 1.2 Product Description Generator
- Prompt
- Function

In [ ]:
# --- Product Description Prompt v1 (Zero-Shot) ---

product_prompt_v1 = """
Create a product description for a wireless mouse that costs $29.99.
"""


In [ ]:
def run_generation_test(prompt, runs=15, log_file=None, label="Generation Test"):
    print(f"\nRunning {label} - {runs} iterations...\n")
    
    results = []
    structure_matches = 0
    
    for i in range(runs):
        result = call_openai(prompt).strip()
        results.append(result)
        print(f"\nRun {i+1}:\n{result}\n")
        
        # Structural checks
        has_headline = "Headline:" in result
        has_description = "Description:" in result
        has_features = "Key Features:" in result
        bullet_count = result.count("- ")
        
        if has_headline and has_description and has_features and bullet_count == 3:
            structure_matches += 1
    
    # Exact match
    baseline = results[0]
    exact_matches = sum(1 for r in results if r == baseline)
    exact_consistency = (exact_matches / runs) * 100
    
    # Structural consistency
    structural_consistency = (structure_matches / runs) * 100
    
    print("Exact-Match Consistency:", f"{exact_consistency:.2f}%")
    print("Structural Consistency:", f"{structural_consistency:.2f}%")
    
    if log_file:
        with open(log_file, "a", encoding="utf-8") as f:
            f.write(f"\n--- {label} ({runs} runs) ---\n")
            for i, res in enumerate(results, 1):
                f.write(f"\nRun {i}:\n{res}\n")
            f.write(f"\nExact-Match Consistency: {exact_consistency:.2f}%\n")
            f.write(f"Structural Consistency: {structural_consistency:.2f}%\n")
            f.write("=" * 60 + "\n")
    
    return exact_consistency, structural_consistency


## 1.3 Data Extraction
- Prompt
- Function (for V1)
- Function (for V2 & V3, with JSON)

In [ ]:
## ==== TODOD: MODIFY PROMPT ===

# --- Data Extraction Prompt v1 (Zero-Shot) ---

extraction_prompt_v1 = """
Extract information from this customer feedback:
"I ordered item #12345 on March 15th. The delivery was fast but the packaging was damaged."
"""


In [ ]:
def run_unstructured_extraction_test(prompt, runs=15, log_file=None, label="Extraction v1"):
    print(f"\nRunning {label} - {runs} iterations...\n")
    
    results = []
    
    for i in range(runs):
        result = call_openai(prompt).strip()
        results.append(result)
        print(f"\nRun {i+1}:\n{result}\n")
    
    baseline = results[0]
    exact_matches = sum(1 for r in results if r == baseline)
    exact_consistency = (exact_matches / runs) * 100
    
    print("Exact-Match Consistency:", f"{exact_consistency:.2f}%")
    
    if log_file:
        with open(log_file, "a", encoding="utf-8") as f:
            f.write(f"\n--- {label} ({runs} runs) ---\n")
            for i, res in enumerate(results, 1):
                f.write(f"\nRun {i}:\n{res}\n")
            f.write(f"\nExact-Match Consistency: {exact_consistency:.2f}%\n")
            f.write("=" * 60 + "\n")
    
    return exact_consistency


In [ ]:
import json

def run_extraction_test(prompt, runs=15, log_file=None, label="Extraction Test"):
    print(f"\nRunning {label} - {runs} iterations...\n")
    
    results = []
    json_structure_matches = 0
    
    required_fields = {"order_id", "order_date", "delivery_speed", "packaging_condition"}
    
    for i in range(runs):
        result = call_openai(prompt).strip()
        results.append(result)
        print(f"\nRun {i+1}:\n{result}\n")
        
        try:
            parsed = json.loads(result)
            if set(parsed.keys()) == required_fields:
                json_structure_matches += 1
        except:
            pass
    
    # Exact match
    baseline = results[0]
    exact_matches = sum(1 for r in results if r == baseline)
    exact_consistency = (exact_matches / runs) * 100
    
    # Structural consistency
    structural_consistency = (json_structure_matches / runs) * 100
    
    print("Exact-Match Consistency:", f"{exact_consistency:.2f}%")
    print("Structural Consistency:", f"{structural_consistency:.2f}%")
    
    if log_file:
        with open(log_file, "a", encoding="utf-8") as f:
            f.write(f"\n--- {label} ({runs} runs) ---\n")
            for i, res in enumerate(results, 1):
                f.write(f"\nRun {i}:\n{res}\n")
            f.write(f"\nExact-Match Consistency: {exact_consistency:.2f}%\n")
            f.write(f"Structural Consistency: {structural_consistency:.2f}%\n")
            f.write("=" * 60 + "\n")
    
    return exact_consistency, structural_consistency


# STEP 2: Run V1 (1, 5. 10. 15 times)

## 2.1 – Sentiment Analysis (Version 1)

This version defines a simple zero-shot prompt with no structural constraints.

Purpose:
- Establish a baseline classification behavior.
- Observe label variability.
- Detect taxonomy drift.
- Measure consistency using repeated runs.

Execution is handled by the reusable `run_classification_test()` function,
which:

- Runs the prompt multiple times
- Computes exact-match consistency
- Logs results to a `.txt` file
- Prints consistency metrics in the notebook

This separates prompt definition from evaluation logic,
ensuring cleaner experimental design.


### STEP 2.1.1: Sentiment v1 – Zero-Shot Baseline (1 Runs) ---

In [ ]:
# --- STEP 2.1.1: Sentiment v1 – Zero-Shot Baseline (1 Runs) ---

# Run x 1
run_classification_test(
    prompt=sentiment_prompt_v1,
    runs=1,
    log_file="sentiment_v1_results.txt",
    label="Sentiment v1 – Zero-Shot Baseline"
)


In [ ]:
# --- STEP 2.1.2: Sentiment v1 – Zero-Shot Baseline (5 Runs) ---

# Run x 5
run_classification_test(
    prompt=sentiment_prompt_v1,
    runs=5,
    log_file="sentiment_v1_results.txt",
    label="Sentiment v1 – Zero-Shot Baseline"
)

### STEP 2.1.3: Sentiment v1 – Zero-Shot Baseline (10 Runs) ---

In [ ]:
# --- STEP 2.1.3: Sentiment v1 – Zero-Shot Baseline (10 Runs) ---

# Run x 10
run_classification_test(
    prompt=sentiment_prompt_v1,
    runs=10,
    log_file="sentiment_v1_results.txt",
    label="Sentiment v1 – Zero-Shot Baseline"
)

### STEP 2.1.4: Sentiment v1 – Zero-Shot Baseline (15 Runs) ---

In [ ]:
# --- STEP 2.1.4: Sentiment v1 – Zero-Shot Baseline (15 Runs) ---
# Run x 15
run_classification_test(
    prompt=sentiment_prompt_v1,
    runs=15,
    log_file="sentiment_v1_results.txt",
    label="Sentiment v1 – Zero-Shot Baseline"
)

### STEP 2.1.5 - Sentiment Analysis – Failure Analysis (v1 Zero-Shot)

<table border="1" cellpadding="8" cellspacing="0">
  <tr>
    <th>No of Runs</th>
    <th>Consistency</th>
    <th>Variations</th>
    <th>Notes</th>
  </tr>
  <tr>
    <td>5</td>
    <td>80%</td>
    <td>
      Minor wording differences; 
      "is a positive product review" vs 
      "is a positive product review and feedback"
    </td>
    <td>
      Semantically consistent but no fixed format. 
      Early signs of instability.
    </td>
  </tr>
  <tr>
    <td>10</td>
    <td>60%</td>
    <td>
      Added structural variation ("can be classified as..."); 
      markdown formatting; expanded labels.
    </td>
    <td>
      Increased format drift and label expansion. 
      Automation risk becomes clear.
    </td>
  </tr>
  <tr>
    <td>15</td>
    <td>53.33%</td>
    <td>
      Multiple unique responses; bold formatting; 
      slash-separated labels; expanded category wording.
    </td>
    <td>
      Clear production instability. 
      Zero-shot prompt lacks constraints and deterministic output.
    </td>
  </tr>
</table>


#### 1️⃣ Format Consistency

Responses are **not consistent in format**.

While all responses follow a sentence structure, wording varies:

- “is a positive product review”
- “can be classified as…”
- Use of bold formatting (`**`)
- Slash-separated labels

There is no fixed schema or constrained output format.

---

#### 2️⃣ Content Consistency

Responses are **semantically consistent**.

All outputs correctly identify the sentiment as positive.  
However, the surface form (exact wording) varies significantly.

---

#### 3️⃣ Types of Variations Observed

- Sentence structure variation  
- Verb tense variation (“is” vs “can be classified as”)  
- Label expansion (“Positive Feedback / Customer Satisfaction”)  
- Formatting drift (bold markdown `**`)  
- Additional descriptive wording  

---

#### 4️⃣ Do More Runs Reveal Additional Failure Patterns?

Yes.

As the number of runs increased:

- **5 runs** → minor wording variation  
- **10 runs** → structural variation and formatting drift  
- **15 runs** → more label expansion and markdown usage  

Consistency decreased as iterations increased, revealing instability.

---

#### 5️⃣ Specific Failure Patterns Identified

- ❌ No fixed label set (model invents variations)  
- ❌ No strict output constraint (full sentences instead of labels)  
- ❌ Formatting drift (markdown formatting appears randomly)  
- ❌ Label drift (expanded sentiment categories)  
- ❌ Exact-match consistency decreases with scale  

---

#### 6️⃣ Evidence of Instability

- Low consistency percentages (down to 53.33%)  
- Multiple unique responses for the same input  
- Increased variation as sample size increases  

This demonstrates that the zero-shot prompt is **not production-ready**.



## 2.2 – Product Description (Version 1 – Zero-Shot Baseline)

This version defines a simple zero-shot prompt with no structural or formatting constraints.

Purpose:
- Establish baseline generation behavior.
- Observe variability in wording and structure.
- Identify format drift across runs.
- Measure consistency using repeated runs.

Execution is handled by the reusable `run_generation_test()` function,
which:

- Runs the prompt multiple times
- Computes exact-match consistency
- Evaluates structural consistency (if applicable)
- Logs results to a `.txt` file
- Prints consistency metrics in the notebook

This separates prompt definition from evaluation logic,
ensuring cleaner experimental design and reproducible testing.


### STEP 2.2.1: Product v1 – Zero-Shot Baseline (1 Runs) ---

In [ ]:
# --- STEP 2.2.1: Product v1 – Zero-Shot Baseline (1 Runs) ---

run_generation_test(
    prompt=product_prompt_v1,
    runs=1,
    log_file="product_v1_results.txt",
    label="Product v1 – Zero-Shot Baseline"
)


### STEP 2.2.2: Product v1 – Zero-Shot Baseline (5 Runs) ---

In [ ]:
# --- STEP 2.2.2: Product v1 – Zero-Shot Baseline (5 Runs) ---

run_generation_test(
    prompt=product_prompt_v1,
    runs=5,
    log_file="product_v1_results.txt",
    label="Product v1 – Zero-Shot Baseline"
)


### STEP 2.2.3: Product v1 – Zero-Shot Baseline (10 Runs) ---

In [ ]:
# --- STEP 2.2.3: Product v1 – Zero-Shot Baseline (10 Runs) ---

run_generation_test(
    prompt=product_prompt_v1,
    runs=10,
    log_file="product_v1_results.txt",
    label="Product v1 – Zero-Shot Baseline"
)


#### STEP 2.2.4: Product v1 – Zero-Shot Baseline (15 Runs) ---

In [ ]:
# --- STEP 2.2.4: Product v1 – Zero-Shot Baseline (15 Runs) ---

run_generation_test(
    prompt=product_prompt_v1,
    runs=15,
    log_file="product_v1_results.txt",
    label="Product v1 – Zero-Shot Baseline"
)


### STEP 2.2.5 - Product Description – Failure Analysis (v1 Zero-Shot)

## 📊 Comprehensive Failure Analysis Table (HTML)

<table border="1" cellpadding="8" cellspacing="0">
  <tr>
    <th>No of Runs</th>
    <th>Consistency</th>
    <th>Variations</th>
    <th>Notes</th>
  </tr>
  <tr>
    <td>1</td>
    <td>100%</td>
    <td>Single generated description</td>
    <td>No variability observed with one run. Not meaningful for reliability evaluation.</td>
  </tr>
  <tr>
    <td>5</td>
    <td>40%</td>
    <td>
      Minor wording differences ("workspace" vs "use"),
      different closing lines,
      slight phrasing adjustments
    </td>
    <td>
      High semantic similarity but exact wording changes reduce consistency.
      No structural constraints defined.
    </td>
  </tr>
  <tr>
    <td>10</td>
    <td>10%</td>
    <td>
      Increased phrasing variation,
      different CTA styles,
      reordered clauses,
      synonym substitutions
    </td>
    <td>
      Significant drop in exact-match consistency.
      Creativity increases instability.
    </td>
  </tr>
  <tr>
    <td>15</td>
    <td>40%</td>
    <td>
      Multiple recurring variants,
      synonym swaps ("functionality", "performance", "affordability"),
      minor sentence restructuring
    </td>
    <td>
      Variants cluster into similar patterns but remain non-deterministic.
      Structural consistency remains 0%.
    </td>
  </tr>
</table>

---

#### 1️⃣ Format Consistency

Responses are **not consistent in exact format**, but structurally similar.

All outputs follow a marketing-style paragraph structure. However:

- Closing call-to-action varies  
- Synonyms are swapped frequently  
- Sentence phrasing changes  
- Some clauses are reordered  

There is **no enforced structure, length constraint, or formatting requirement**, which leads to variation.

---

#### 2️⃣ Content Consistency

Responses are **highly consistent semantically**.

All descriptions:
- Mention price ($29.99)
- Highlight wireless functionality
- Emphasize comfort and precision
- Include battery life
- Use promotional marketing tone

However, exact wording varies significantly, reducing exact-match consistency.

---

#### 3️⃣ Types of Variations Observed

- Synonym variation ("performance" vs "functionality" vs "affordability")
- CTA variation (“Upgrade your setup today!” variations)
- Word substitution (“workspace” vs “use”)
- Minor clause reordering
- Slight punctuation and dash usage changes
- Tone intensity shifts

---

#### 4️⃣ Do More Runs Reveal Additional Failure Patterns?

Yes.

- **5 runs** → minor wording changes  
- **10 runs** → significant drop in exact-match consistency (10%)  
- **15 runs** → multiple recurring variants form, but still unstable  

As iterations increase:
- Exact-match consistency drops sharply
- Creativity increases
- Structural consistency remains 0%

Unlike classification, instability here is due to **generative variability**, not label drift.

---

#### 5️⃣ Specific Failure Patterns Identified

- ❌ No defined length constraint  
- ❌ No fixed structure (e.g., bullet points, template format)  
- ❌ No required section breakdown  
- ❌ No deterministic closing format  
- ❌ Synonym variability reduces exact-match reliability  
- ❌ Structural consistency remains 0% across all runs  

---

#### 6️⃣ Evidence of Instability

- Exact-match consistency drops to 10% at 10 runs  
- Multiple unique but semantically similar outputs  
- Structural consistency = 0% across all tests  
- Increased variation with larger sample sizes  

This demonstrates that while semantically strong, the zero-shot generative prompt lacks production-level determinism and structural control.


## 2.3 – Data Extraction (Version 1)

This version defines a simple zero-shot extraction prompt with no structured output constraints.

Purpose:
- Establish baseline extraction behavior.
- Observe variability in extracted information.
- Identify structural drift in free-form responses.
- Measure consistency using repeated runs.

Execution is handled by the reusable `run_unstructured_extraction_test()` function,
which:

- Runs the prompt multiple times
- Computes exact-match consistency
- Logs results to a `.txt` file
- Prints consistency metrics in the notebook

Since this version does not enforce a schema, 
structural consistency is not evaluated at this stage.

This separates prompt definition from evaluation logic,
ensuring cleaner experimental design and controlled baseline measurement.


### STEP 2.3.1: Extraction v1 – Zero-Shot Baseline (1 Runs) ---

In [ ]:
# --- STEP 2.3.1: Extraction v1 – Zero-Shot Baseline (1 Runs) ---

run_unstructured_extraction_test(
    prompt=extraction_prompt_v1,
    runs=1,
    log_file="extraction_v1_results.txt",
    label="Extraction v1 – Zero-Shot Baseline"
)


### STEP 2.3.2: Extraction v1 – Zero-Shot Baseline (5 Runs) ---

In [ ]:
# --- STEP 2.3.2: Extraction v1 – Zero-Shot Baseline (5 Runs) ---

run_unstructured_extraction_test(
    prompt=extraction_prompt_v1,
    runs=5,
    log_file="extraction_v1_results.txt",
    label="Extraction v1 – Zero-Shot Baseline"
)


### STEP 2.3.3: Extraction v1 – Zero-Shot Baseline (10 Runs) ---

In [ ]:
# --- STEP 2.3.3: Extraction v1 – Zero-Shot Baseline (10 Runs) ---

run_unstructured_extraction_test(
    prompt=extraction_prompt_v1,
    runs=10,
    log_file="extraction_v1_results.txt",
    label="Extraction v1 – Zero-Shot Baseline"
)


### STEP 2.3.4: Extraction v1 – Zero-Shot Baseline (15 Runs) ---

In [ ]:
# --- STEP 2.3.4: Extraction v1 – Zero-Shot Baseline (15 Runs) ---

run_unstructured_extraction_test(
    prompt=extraction_prompt_v1,
    runs=15,
    log_file="extraction_v1_results.txt",
    label="Extraction v1 – Zero-Shot Baseline"
)


### STEP 2.3.5 Data Extraction – Failure Analysis (v1 Zero-Shot)

#### 📊 Comprehensive Failure Analysis Table (HTML)

<table border="1" cellpadding="8" cellspacing="0">
  <tr>
    <th>No of Runs</th>
    <th>Consistency</th>
    <th>Variations</th>
    <th>Notes</th>
  </tr>
  <tr>
    <td>1</td>
    <td>100%</td>
    <td>Single structured bullet output with bold formatting and intro sentence</td>
    <td>Not meaningful for reliability evaluation. No variation observed.</td>
  </tr>
  <tr>
    <td>5</td>
    <td>20%</td>
    <td>
      Variation in field names ("Delivery Speed" vs "Delivery Experience"),
      optional "#" in order number,
      presence/absence of intro sentence,
      bold formatting inconsistently applied
    </td>
    <td>
      Rapid drop in exact-match consistency.
      Field naming instability appears immediately.
    </td>
  </tr>
  <tr>
    <td>10</td>
    <td>40%</td>
    <td>
      Mixed structured styles (bold vs plain text),
      intro text sometimes included,
      inconsistent use of "#",
      alternating field labels
    </td>
    <td>
      Formatting drift and schema inconsistency persist.
      Multiple response templates emerge.
    </td>
  </tr>
  <tr>
    <td>15</td>
    <td>40%</td>
    <td>
      Recurring template clusters,
      field name variation,
      markdown formatting inconsistencies,
      inconsistent order number format
    </td>
    <td>
      Output stabilizes into a few dominant patterns,
      but no deterministic schema.
      Not machine-parse safe.
    </td>
  </tr>
</table>



#### 1️⃣ Format Consistency

Responses are **not consistent in format**.

Observed inconsistencies include:

- Introductory sentence sometimes included (“Certainly! Here is…”)
- Bold markdown formatting sometimes applied
- Bullet formatting varies slightly
- Field names vary
- Presence or absence of "#" in order number

There is no enforced schema or structured output format (e.g., JSON), resulting in instability.

---
#### 2️⃣ Content Consistency

Responses are **semantically consistent**.

All runs correctly extract:

- Order number
- Order date
- Delivery information
- Packaging condition

However, the representation of that data varies significantly, affecting reliability.

---

#### 3️⃣ Types of Variations Observed

- Field name variation (“Delivery Speed” vs “Delivery Experience”)
- Formatting drift (bold vs plain text)
- Optional introductory explanation text
- Inconsistent inclusion of "#" in order number
- Minor structural formatting changes

---

#### 4️⃣ Do More Runs Reveal Additional Failure Patterns?

Yes.

- **5 runs** → immediate schema variation and formatting drift  
- **10 runs** → multiple output templates emerge  
- **15 runs** → patterns cluster but remain non-deterministic  

Unlike sentiment classification, instability here directly affects structured data integrity, making automation risky.

---

#### 5️⃣ Specific Failure Patterns Identified

- ❌ No fixed schema definition  
- ❌ Field name inconsistency  
- ❌ Inconsistent order number formatting (# vs no #)  
- ❌ Optional explanatory intro text  
- ❌ Markdown formatting variability  
- ❌ Not JSON-structured or machine-parseable  

---

#### 6️⃣ Evidence of Instability

- Consistency drops to 20% at 5 runs  
- Multiple unique structured outputs  
- Schema instability across iterations  
- Formatting drift persists at scale  

This demonstrates that the zero-shot extraction prompt is **not production-ready for structured data pipelines**.



# STEP 3: Version 2
- Modified prompt
- Run 15 times

## STEP 3.1 -  Sentiment Analysis (Version 2 – Structured Constraints) 

This version improves the baseline prompt by enforcing explicit output constraints.

Purpose:
- Eliminate taxonomy drift.
- Prevent explanatory text in responses.
- Enforce strict label control.
- Improve production reliability.

Improvements introduced:

- Explicit allowed label set (Positive, Negative, Neutral)
- Forced single-word output
- Explicit prohibition of explanations

Execution is handled by the reusable `run_classification_test()` function,
which:

- Runs the prompt multiple times
- Computes exact-match consistency
- Logs results to a `.txt` file
- Prints consistency metrics in the notebook

This maintains separation between prompt design and evaluation logic.


### STEP 3.1.1 - Sentiment Prompt v2 (Structured Constraints) ---

In [ ]:
# --- STEP 3.1.1 - Sentiment Prompt v2 (Structured Constraints) ---

sentiment_prompt_v2 = """
Classify the sentiment of the following customer message.

Customer Message:
"I love this product! It's exactly what I needed."

Respond with exactly ONE word:
Positive
Negative
Neutral

Do not include any explanation or additional text.
"""


### STEP 3.1.2: Sentiment v2 – Structured Constraints (15 Runs) ---

In [ ]:
# --- STEP 3.1.2: Sentiment v2 – Structured Constraints (15 Runs) ---

run_classification_test(
    prompt=sentiment_prompt_v2,
    runs=15,
    log_file="sentiment_v2_results.txt",
    label="Sentiment v2 – Structured Constraints"
)


### STEP 3.1.3 - Sentiment Analysis – Improvement Evaluation (v2 Structured Constraints)

#### 📊 Comparison Table (HTML)

<table border="1" cellpadding="8" cellspacing="0">
  <tr>
    <th>Version</th>
    <th>N Runs</th>
    <th>Consistency</th>
    <th>Variations</th>
    <th>Notes</th>
  </tr>
  <tr>
    <td>V1 – Zero-Shot</td>
    <td>15</td>
    <td>53.33%</td>
    <td>
      Sentence structure variation;
      label expansion;
      markdown formatting drift;
      explanatory wording differences
    </td>
    <td>
      Semantically correct but structurally unstable.
      No label constraints or fixed output schema.
      Not production-safe.
    </td>
  </tr>
  <tr>
    <td>V2 – Structured Constraints</td>
    <td>15</td>
    <td>100%</td>
    <td>
      No variations observed.
      Single-word output ("Positive")
    </td>
    <td>
      Explicit label set and single-word constraint eliminated format drift.
      Deterministic and production-ready.
    </td>
  </tr>
</table>

#### 📌 Summary

Introducing structured constraints had a dramatic impact on reliability.

V1 showed semantic correctness but structural instability, with consistency decreasing as the number of runs increased. The lack of explicit formatting rules and label constraints allowed the model to generate varied sentence structures.

V2 enforced:

- A fixed label set
- A strict single-word response requirement
- Clear output constraints
- This resulted in 100% consistency across 15 runs, demonstrating that classification tasks respond extremely well to deterministic formatting and constrained output design.


## STEP 3.2 – Product Description (Version 2 – Structured Constraints)

This version improves the baseline generation prompt by introducing explicit structural constraints.

Purpose:
- Enforce stable output formatting.
- Prevent section drift across runs.
- Control layout and bullet structure.
- Improve production readiness.

Improvements introduced:

- Explicit format template (Headline, Description, Key Features)
- Section headings enforced
- Word limit guidance
- Exact number of bullet points (3)
- Professional tone guidance

Execution is handled by the reusable `run_generation_test()` function,
which:

- Runs the prompt multiple times
- Computes exact-match consistency
- Computes structural consistency
- Logs results to a `.txt` file
- Prints consistency metrics in the notebook

This maintains clean separation between prompt design and evaluation logic.


### STEP 3.2.1 -Product Prompt v2 (Structured Constraints) ---

In [ ]:
# --- STEP 3.2.1 -Product Prompt v2 (Structured Constraints) ---

product_prompt_v2 = """
Create a product description for a wireless mouse that costs $29.99.

Requirements:

1. Start with a short headline (max 8 words).
2. Write a 2–3 sentence paragraph (max 80 words total).
3. Include exactly 3 bullet points listing key features.
4. Use a professional and persuasive tone.
5. Do not exceed 120 words total.

Structure the output exactly like this:

Headline:
<text>

Description:
<text>

Key Features:
- <feature 1>
- <feature 2>
- <feature 3>
"""


### STEP 3.2.2: Product v2 – Structured Constraints (15 Runs) ---

In [ ]:
# --- STEP 3.2.2: Product v2 – Structured Constraints (15 Runs) ---

run_generation_test(
    prompt=product_prompt_v2,
    runs=15,
    log_file="product_v2_results.txt",
    label="Product v2 – Structured Constraints"
)


### STEP 3.2.3 - Product Description – Improvement Evaluation (v2 Structured Constraints)

#### 📊 Comparison Table (HTML)

<table border="1" cellpadding="8" cellspacing="0">
  <tr>
    <th>Version</th>
    <th>N Runs</th>
    <th>Exact-Match Consistency</th>
    <th>Structural Consistency</th>
    <th>Variations</th>
    <th>Notes</th>
  </tr>
  <tr>
    <td>V1 – Zero-Shot</td>
    <td>15</td>
    <td>40%</td>
    <td>0%</td>
    <td>
      Synonym substitutions; CTA variation; clause reordering;
      minor wording shifts; no fixed structure
    </td>
    <td>
      Semantically strong but structurally uncontrolled.
      No enforced sections or formatting template.
      High creative drift.
    </td>
  </tr>
  <tr>
    <td>V2 – Structured Constraints</td>
    <td>15</td>
    <td>40%</td>
    <td>100%</td>
    <td>
      Minor description wording changes;
      feature wording variation;
      range units (10 meters vs 30 feet);
      battery feature variations
    </td>
    <td>
      Clear section template (Headline, Description, Key Features)
      fully stabilized structure.
      Content remains variable but format is deterministic.
    </td>
  </tr>
</table>

####📌 Summary

Introducing structured formatting dramatically improved structural reliability, but not exact-match determinism.

In V1:

- Outputs were creative and semantically strong
- No consistent structure
- Structural consistency = 0%

In V2:

- Headline / Description / Key Features sections enforced
- Structural consistency increased to 100%
- Exact-match consistency remained at 40% due to natural language variation

This demonstrates an important insight:

For generative tasks, structural constraints improve formatting reliability, but content variability remains unless stronger constraints (e.g., length limits, fixed feature list, few-shot examples) are introduced.

V2 is structurally production-ready, but not fully deterministic at the content level.

## STEP 3.3 – Data Extraction (Version 2 – Structured JSON Output)

This version improves the baseline extraction prompt by enforcing a strict JSON schema.

Purpose:
- Guarantee machine-readable output.
- Eliminate structural drift.
- Ensure compatibility with downstream systems.
- Improve production reliability.

Improvements introduced:

- Explicit required fields
- Fixed JSON schema
- No additional fields allowed
- Double-quote enforcement
- Null handling rule

Execution is handled by the reusable `run_extraction_test()` function,
which:

- Runs the prompt multiple times
- Computes exact-match consistency
- Computes JSON structural consistency
- Logs results to a `.txt` file
- Prints consistency metrics in the notebook

This maintains separation between prompt design and evaluation logic.


### STEP 3.3.1 - Extraction Prompt v2 (Structured JSON) ---

In [ ]:
# --- STEP 3.3.1 - Extraction Prompt v2 (Structured JSON) ---

extraction_prompt_v2 = """
Extract structured information from the following customer feedback.

Customer Feedback:
"I ordered item #12345 on March 15th. The delivery was fast but the packaging was damaged."

Return the output in valid JSON format with EXACTLY the following fields:

{
  "order_id": "",
  "order_date": "",
  "delivery_speed": "",
  "packaging_condition": ""
}

Rules:
- Do not add extra fields.
- Use double quotes.
- If information is missing, use null.
- Return only valid JSON.
"""


### STEP 3.3.2: Extraction v2 – Structured JSON (15 Runs) ---

In [ ]:
# --- STEP 3.3.2: Extraction v2 – Structured JSON (15 Runs) ---

run_extraction_test(
    prompt=extraction_prompt_v2,
    runs=15,
    log_file="extraction_v2_results.txt",
    label="Extraction v2 – Structured JSON"
)


### STEP 3.2.3 - Data Extraction – Improvement Evaluation (v2 Structured JSON)

#### 📊 Comparison Table (HTML)

<table border="1" cellpadding="8" cellspacing="0">
  <tr>
    <th>Version</th>
    <th>N Runs</th>
    <th>Exact-Match Consistency</th>
    <th>Structural Consistency</th>
    <th>Variations</th>
    <th>Notes</th>
  </tr>
  <tr>
    <td>V1 – Zero-Shot</td>
    <td>15</td>
    <td>40%</td>
    <td>0%</td>
    <td>
      Field name variation ("Delivery Speed" vs "Delivery Experience");
      optional "#" in order number;
      markdown formatting drift;
      optional introductory text
    </td>
    <td>
      Semantically correct but schema unstable.
      Not machine-parse safe.
      Multiple output templates emerged.
    </td>
  </tr>
  <tr>
    <td>V2 – Structured JSON</td>
    <td>15</td>
    <td>100%</td>
    <td>100%</td>
    <td>
      No variations observed.
      Identical JSON structure and values across all runs.
    </td>
    <td>
      Explicit JSON schema and field constraints eliminated drift.
      Fully deterministic and production-ready for data pipelines.
    </td>
  </tr>
</table>

#### 📌 Summary

Introducing a strict JSON schema completely stabilized the extraction task.

In V1:

- Outputs were semantically accurate
- Field names and formatting varied
- Structural consistency was 0%
- Automation risk was high

In V2:

- A fixed JSON schema was enforced
- Field names were standardized
- No introductory text was generated
- All 15 runs produced identical outputs
- Both exact-match and structural consistency reached 100%, demonstrating that structured data extraction tasks benefit dramatically from explicit schema definition and format constraints.

This version is fully production-ready for structured data pipelines.

# STEP 4 - Version 3

## STEP 4.1 – Sentiment Analysis (Version 3 – Few-Shot Prompting)

This version enhances the structured prompt by introducing few-shot examples.

Purpose:
- Reinforce classification boundaries.
- Prevent taxonomy drift.
- Improve semantic alignment.
- Increase robustness under ambiguous input.

Improvements introduced:

- Three labeled examples (Positive, Negative, Neutral)
- Explicit output format enforcement
- Strict single-word response requirement
- No explanation allowed

Execution is handled by the reusable `run_classification_test()` function,
which:

- Runs the prompt multiple times
- Computes exact-match consistency
- Logs results to a `.txt` file
- Prints consistency metrics in the notebook

This preserves separation between prompt engineering and evaluation logic.


### STEP 4.1.1 - Sentiment Prompt v3 (Few-Shot + Structured) ---

In [ ]:
# --- STEP 4.1.1 - Sentiment Prompt v3 (Few-Shot + Structured) ---

sentiment_prompt_v3 = """
You are a sentiment classification system.

Classify customer messages into EXACTLY one of the following categories:
Positive
Negative
Neutral

Respond with ONLY one word from the list above.
Do not provide explanations.

Examples:

Customer Message:
"I absolutely love this! Best purchase I've made."
Output:
Positive

Customer Message:
"This is terrible and completely unusable."
Output:
Negative

Customer Message:
"It works okay, nothing special."
Output:
Neutral

Now classify the following message:

Customer Message:
"I love this product! It's exactly what I needed."

Output:
"""


### STEP 4.1.2: Sentiment v3 – Few-Shot Prompting (15 Runs) ---

In [ ]:
# --- STEP 4.1.2: Sentiment v3 – Few-Shot Prompting (15 Runs) ---

run_classification_test(
    prompt=sentiment_prompt_v3,
    runs=15,
    log_file="sentiment_v3_results.txt",
    label="Sentiment v3 – Few-Shot Prompting"
)

### STEP 4.1.3 – Sentiment Analysis – Improvement Evaluation (v3 Few-Shot Prompting)

#### 📊 Comparison Table (HTML)

<table border="1" cellpadding="8" cellspacing="0">
  <tr>
    <th>Version</th>
    <th>N Runs</th>
    <th>Exact-Match Consistency</th>
    <th>Structural Consistency</th>
    <th>Variations</th>
    <th>Notes</th>
  </tr>
  <tr>
    <td>V1 – Zero-Shot</td>
    <td>15</td>
    <td>53.33%</td>
    <td>0%</td>
    <td>
      Sentence structure variation;
      label expansion;
      markdown formatting drift;
      explanatory wording differences
    </td>
    <td>
      Semantically correct but structurally unstable.
      No fixed label constraint.
      Not production-safe.
    </td>
  </tr>
  <tr>
    <td>V2 – Structured Constraints</td>
    <td>15</td>
    <td>100%</td>
    <td>100%</td>
    <td>
      No variations observed.
      Single-word output.
    </td>
    <td>
      Explicit label set and single-word restriction
      eliminated format drift.
      Fully deterministic.
    </td>
  </tr>
  <tr>
    <td>V3 – Few-Shot Prompting</td>
    <td>15</td>
    <td>100%</td>
    <td>100%</td>
    <td>
      No variations observed.
      Identical single-word output across all runs.
    </td>
    <td>
      Few-shot examples reinforced expected output format.
      Demonstrates high stability and robustness.
    </td>
  </tr>
</table>

#### 📌 Summary

Adding few-shot examples maintained full determinism in the classification task.

In V1:

- Semantic correctness but structural instability  
- Format drift increased with scale  
- Not automation-safe  

In V2:

- Fixed label set and single-word constraint  
- Consistency reached 100%  
- Deterministic output  

In V3:

- Few-shot examples reinforced the expected format  
- Stability remained at 100% across 15 runs  
- Demonstrates that few-shot prompting strengthens format reliability and makes the behavior more robust to variation  

For simple classification tasks, combining label constraints with few-shot examples produces fully production-ready, deterministic outputs.


## STEP 4.2 – Product Description (Version 3 – Few-Shot + Structured)

This version enhances the structured template by incorporating few-shot examples.

Purpose:
- Improve stylistic alignment.
- Reinforce formatting consistency.
- Reduce phrasing drift.
- Maintain structural stability across runs.

Improvements introduced:

- Two high-quality few-shot examples
- Explicit format template (Headline, Description, Key Features)
- Exact bullet count enforcement (3)
- Word limit guidance
- Professional tone reinforcement

Execution is handled by the reusable `run_generation_test()` function,
which:

- Runs the prompt multiple times
- Computes exact-match consistency
- Computes structural consistency
- Logs results to a `.txt` file
- Prints consistency metrics in the notebook

This maintains clean separation between prompt design and evaluation logic.


### STEP 4.2.1 - Product Prompt v3 (Few-Shot + Structured) ---

In [ ]:
# --- STEP 4.2.1 - Product Prompt v3 (Few-Shot + Structured) ---

product_prompt_v3 = """
You are a professional e-commerce copywriter.

Follow the EXACT format shown in the examples below.

EXAMPLE 1

Headline:
Compact Power for Everyday Use

Description:
This lightweight portable charger keeps your devices powered throughout the day. Designed for convenience and durability, it delivers fast and reliable charging wherever you go.

Key Features:
- 10,000mAh high-capacity battery
- Fast USB-C charging
- Slim and travel-friendly design


EXAMPLE 2

Headline:
Crystal Clear Sound Anywhere

Description:
Experience immersive audio with this compact Bluetooth speaker. Built for portability and performance, it delivers rich sound quality indoors and outdoors.

Key Features:
- Wireless Bluetooth connectivity
- 12-hour battery life
- Water-resistant exterior


NOW CREATE A PRODUCT DESCRIPTION FOR:

Product: Wireless Mouse
Price: $29.99

Requirements:
- Follow the exact same structure.
- Use exactly 3 bullet points.
- Keep total word count under 120 words.
- Use a professional and persuasive tone.
- Do not add extra sections.
"""


### STEP 4.2.2: Product v3 – Few-Shot + Structured (15 Runs) ---

In [ ]:
# --- STEP 4.2.2: Product v3 – Few-Shot + Structured (15 Runs) ---

run_generation_test(
    prompt=product_prompt_v3,
    runs=15,
    log_file="product_v3_results.txt",
    label="Product v3 – Few-Shot + Structured"
)


### STEP 4.3.3 – Product Description – Improvement Evaluation (v3 Few-Shot + Structured)

#### 📊 Comparison Table (HTML)

<table border="1" cellpadding="8" cellspacing="0">
  <tr>
    <th>Version</th>
    <th>N Runs</th>
    <th>Exact-Match Consistency</th>
    <th>Structural Consistency</th>
    <th>Variations</th>
    <th>Notes</th>
  </tr>
  <tr>
    <td>V1 – Zero-Shot</td>
    <td>15</td>
    <td>40%</td>
    <td>0%</td>
    <td>
      Synonym substitutions; CTA variation;
      clause reordering; no enforced sections
    </td>
    <td>
      Creative but structurally unstable.
      No deterministic formatting.
    </td>
  </tr>
  <tr>
    <td>V2 – Structured Constraints</td>
    <td>15</td>
    <td>40%</td>
    <td>100%</td>
    <td>
      Minor wording shifts in description;
      feature wording variation;
      unit changes (meters vs feet)
    </td>
    <td>
      Section template stabilized structure
      but content variability remained.
    </td>
  </tr>
  <tr>
    <td>V3 – Few-Shot + Structured</td>
    <td>15</td>
    <td>26.67%</td>
    <td>100%</td>
    <td>
      Headline variation;
      description paraphrasing;
      feature substitutions;
      unit variation (meters, feet, 33ft);
      sensor/battery variations
    </td>
    <td>
      Structure fully stable.
      Few-shot improved stylistic alignment,
      but generative variability remains high.
    </td>
  </tr>
</table>

#### 📌 Summary

Adding few-shot examples reinforced stylistic consistency but did not increase exact-match determinism.

In V1:

- High creativity  
- No structural control  
- Structural consistency = 0%  

In V2:

- Template (Headline / Description / Key Features) stabilized format  
- Structural consistency = 100%  
- Content still varied  

In V3:

- Few-shot examples improved tone alignment  
- Structure remained fully stable (100%)  
- Exact-match consistency decreased to 26.67% due to increased creative diversity  

Key Insight:

For generative marketing tasks, few-shot prompting improves stylistic coherence but does not enforce deterministic outputs. Structural constraints control format, but content variability remains unless tighter constraints (fixed feature list, word count, controlled vocabulary) are introduced.

This version is structurally production-ready but not fully deterministic at the content level.


## STEP 4.3 – Data Extraction (Version 3 – Chain-of-Thought + Structured Output)

This version enhances the structured JSON prompt by introducing hidden Chain-of-Thought reasoning.

Purpose:
- Improve reasoning stability.
- Reduce variability under ambiguity.
- Maintain strict schema enforcement.
- Increase robustness while preserving machine-readable output.

Improvements introduced:

- Internal step-by-step reasoning
- Explicit reasoning targets (order ID, date, delivery speed, packaging condition)
- Strict JSON schema enforcement
- No reasoning allowed in final output

Execution is handled by the reusable `run_extraction_test()` function,
which:

- Runs the prompt multiple times
- Computes exact-match consistency
- Computes JSON structural consistency
- Logs results to a `.txt` file
- Prints consistency metrics in the notebook

To maintain experimental consistency, the same customer feedback used in
Versions 1 and 2 is used for evaluation.


### STEP 4.3.1 - Extraction Prompt v3 (CoT + JSON, Same Input as v1/v2) ---

In [ ]:
# --- STEP 4.3.1 - Extraction Prompt v3 (CoT + JSON, Same Input as v1/v2) ---

extraction_prompt_v3 = """
You are an information extraction system.

Extract structured data from the customer feedback below.

Customer Feedback:
"I ordered item #12345 on March 15th. The delivery was fast but the packaging was damaged."

First, think step by step about:
- Order ID
- Order date
- Delivery speed
- Packaging condition

Then return ONLY the final answer in valid JSON format with EXACTLY these fields:

{
  "order_id": "",
  "order_date": "",
  "delivery_speed": "",
  "packaging_condition": ""
}

Rules:
- Do not include your reasoning in the output.
- Return only valid JSON.
- Use double quotes.
- If information is missing, use null.
"""


### STEP 4.3.2: Extraction v3 – CoT + JSON (15 Runs) ---

In [ ]:
# --- STEP 4.3.2: Extraction v3 – CoT + JSON (15 Runs) ---

run_extraction_test(
    prompt=extraction_prompt_v3,
    runs=15,
    log_file="extraction_v3_results.txt",
    label="Extraction v3 – CoT + JSON"
)


### STEP 4.3.3 – Data Extraction – Improvement Evaluation (v3 CoT + JSON)

#### 📊 Comparison Table (HTML)

<table border="1" cellpadding="8" cellspacing="0">
  <tr>
    <th>Version</th>
    <th>N Runs</th>
    <th>Exact-Match Consistency</th>
    <th>Structural Consistency</th>
    <th>Variations</th>
    <th>Notes</th>
  </tr>
  <tr>
    <td>V1 – Zero-Shot</td>
    <td>15</td>
    <td>40%</td>
    <td>0%</td>
    <td>
      Field name variation;
      optional "#" in order number;
      markdown formatting drift;
      introductory explanation text
    </td>
    <td>
      Semantically correct but schema unstable.
      Multiple output templates.
      Not safe for automated parsing.
    </td>
  </tr>
  <tr>
    <td>V2 – Structured JSON</td>
    <td>15</td>
    <td>100%</td>
    <td>100%</td>
    <td>
      No variations observed.
      Identical JSON output across runs.
    </td>
    <td>
      Explicit schema eliminated formatting drift.
      Fully deterministic and production-ready.
    </td>
  </tr>
  <tr>
    <td>V3 – CoT + JSON</td>
    <td>15</td>
    <td>100%</td>
    <td>100%</td>
    <td>
      No variations observed.
      Identical structured JSON output.
    </td>
    <td>
      Chain-of-Thought reasoning did not change output format
      but reinforces reasoning robustness.
      Maintains full determinism.
    </td>
  </tr>
</table>

#### 📌 Summary

Adding structured JSON in V2 completely stabilized the extraction task.

In V1:

- Field names and formatting varied  
- Schema inconsistency created automation risk  
- Structural consistency was 0%  

In V2:

- A strict JSON schema was enforced  
- Exact-match and structural consistency reached 100%  
- Fully deterministic and machine-parseable  

In V3:

- Chain-of-Thought reasoning was introduced  
- Output remained 100% consistent  
- Deterministic behavior was preserved  

Key Insight:

For structured data extraction, schema constraints are the dominant stabilizing factor. Chain-of-Thought reasoning may improve reasoning robustness in more complex cases, but determinism is primarily achieved through explicit output structure.

This version is fully production-ready for structured data pipelines.


# STEP 5 Robustness Testing

## STEP 5.1 Robustness Testing – Sentiment Classification (Extended Inputs)

The original customer feedback used in Versions 1 and 2 was relatively straightforward:

"I love this product! It's exactly what I needed."

Because this input was simple and clearly positive, it did not fully test the robustness of the prompts under more realistic conditions.

To strengthen the evaluation, additional tests were conducted using longer and more complex customer feedback, including:

- A detailed positive review
- A detailed negative review
- A nuanced neutral review

These extended inputs included multiple sentences, mixed signals, and richer contextual information, better reflecting real-world customer feedback.

The goal of this robustness test was to:

- Evaluate classification stability under increased complexity
- Detect potential taxonomy drift in Version 1
- Compare consistency and label control between Version 1 and Version 3
- Assess whether structured constraints and few-shot prompting improve reliability under non-trivial input

Execution reused the existing `run_classification_test()` function and appended results to the existing result files to preserve experimental continuity.

This robustness phase complements the baseline comparison by demonstrating how prompt design performs under more realistic and varied conditions.


#### STEP 5.1.1 Robustness Test Inputs (Complex Feedback) ---

In [ ]:
# --- STEP 5.1.1 Robustness Test Inputs (Complex Feedback) ---

complex_positive_feedback = """
I ordered the wireless mouse last week and it arrived earlier than expected.
The packaging was secure and the setup was incredibly simple.
I've been using it daily for work and the battery life is impressive.
Overall, I think it's an excellent value for the price.
"""

complex_negative_feedback = """
I received my wireless mouse three days ago and I'm quite disappointed.
The cursor frequently lags and the build quality feels cheap.
The battery drains much faster than expected.
Overall, I regret this purchase.
"""

complex_neutral_feedback = """
The wireless mouse works as described and connects easily to my laptop.
The design is simple and functional, though nothing stands out.
It performs adequately for everyday tasks.
Overall, it is an average product.
"""


### STEP 5.1.2 - Build Prompts for Robustness Testing (V1) 

In [ ]:
# STEP 5.1.2 - Build Prompts for Robustness Testing (V1) 
def build_sentiment_prompt_v1(feedback):
    return f"""
Classify this customer message:
"{feedback}"
"""

def build_sentiment_prompt_v3(feedback):
    return f"""
You are a sentiment classification system.

Classify customer messages into EXACTLY one of the following categories:
Positive
Negative
Neutral

Respond with ONLY one word from the list above.
Do not provide explanations.

Examples:

Customer Message:
"I absolutely love this! Best purchase I've made."
Output:
Positive

Customer Message:
"This is terrible and completely unusable."
Output:
Negative

Customer Message:
"It works okay, nothing special."
Output:
Neutral

Now classify the following message:

Customer Message:
"{feedback}"

Output:
"""


#### STEP 5.1.3 Sentiment v1 – Robustness Tests (Append) ---

In [ ]:
# --- STEP 5.1.3 Sentiment v1 – Robustness Tests (Append) ---

run_classification_test(
    prompt=f'Classify this customer message:\n"{complex_positive_feedback}"',
    runs=15,
    log_file="sentiment_v1_results.txt",
    label="Sentiment v1 – Robustness (Positive)"
)

run_classification_test(
    prompt=f'Classify this customer message:\n"{complex_negative_feedback}"',
    runs=15,
    log_file="sentiment_v1_results.txt",
    label="Sentiment v1 – Robustness (Negative)"
)

run_classification_test(
    prompt=f'Classify this customer message:\n"{complex_neutral_feedback}"',
    runs=15,
    log_file="sentiment_v1_results.txt",
    label="Sentiment v1 – Robustness (Neutral)"
)


#### STEP 5.1.4  Build Sentiment v3 Robustness Prompts ---

In [ ]:
# ---STEP 5.1.4  Build Sentiment v3 Robustness Prompts ---

sentiment_v3_positive_prompt = f"""
You are a sentiment classification system.

Classify customer messages into EXACTLY one of the following categories:
Positive
Negative
Neutral

Respond with ONLY one word from the list above.
Do not provide explanations.

Examples:

Customer Message:
"I absolutely love this! Best purchase I've made."
Output:
Positive

Customer Message:
"This is terrible and completely unusable."
Output:
Negative

Customer Message:
"It works okay, nothing special."
Output:
Neutral

Now classify the following message:

Customer Message:
"{complex_positive_feedback}"

Output:
"""

sentiment_v3_negative_prompt = f"""
You are a sentiment classification system.

Classify customer messages into EXACTLY one of the following categories:
Positive
Negative
Neutral

Respond with ONLY one word from the list above.
Do not provide explanations.

Examples:

Customer Message:
"I absolutely love this! Best purchase I've made."
Output:
Positive

Customer Message:
"This is terrible and completely unusable."
Output:
Negative

Customer Message:
"It works okay, nothing special."
Output:
Neutral

Now classify the following message:

Customer Message:
"{complex_negative_feedback}"

Output:
"""

sentiment_v3_neutral_prompt = f"""
You are a sentiment classification system.

Classify customer messages into EXACTLY one of the following categories:
Positive
Negative
Neutral

Respond with ONLY one word from the list above.
Do not provide explanations.

Examples:

Customer Message:
"I absolutely love this! Best purchase I've made."
Output:
Positive

Customer Message:
"This is terrible and completely unusable."
Output:
Negative

Customer Message:
"It works okay, nothing special."
Output:
Neutral

Now classify the following message:

Customer Message:
"{complex_neutral_feedback}"

Output:
"""


### STEP 5.1.5  Sentiment v3 – Robustness Tests (Append to Existing File) ---

In [ ]:
# --- STEP 5.1.5  Sentiment v3 – Robustness Tests (Append to Existing File) ---

run_classification_test(
    prompt=sentiment_v3_positive_prompt,
    runs=15,
    log_file="sentiment_v3_results.txt",
    label="Sentiment v3 – Robustness (Positive)"
)

run_classification_test(
    prompt=sentiment_v3_negative_prompt,
    runs=15,
    log_file="sentiment_v3_results.txt",
    label="Sentiment v3 – Robustness (Negative)"
)

run_classification_test(
    prompt=sentiment_v3_neutral_prompt,
    runs=15,
    log_file="sentiment_v3_results.txt",
    label="Sentiment v3 – Robustness (Neutral)"
)


### STEP 5.1.6 – Sentiment Classification – Robustness Evaluation (Extended Inputs)

#### 📊 Robustness Comparison Table (HTML)

<table border="1" cellpadding="8" cellspacing="0">
  <tr>
    <th>Version</th>
    <th>Positive Consistency</th>
    <th>Negative Consistency</th>
    <th>Neutral Consistency</th>
    <th>Notes</th>
  </tr>
  <tr>
    <td>V1 – Zero-Shot</td>
    <td>93.33%</td>
    <td>40.00%</td>
    <td>53.33%</td>
    <td>
      High stability for clearly positive input.
      Significant taxonomy drift for negative and neutral cases.
      Explanatory text and category expansion observed.
    </td>
  </tr>
  <tr>
    <td>V3 – Few-Shot + Constraints</td>
    <td>100.00%</td>
    <td>100.00%</td>
    <td>100.00%</td>
    <td>
      Fully deterministic across all sentiment categories.
      No label drift, no explanatory text.
      Strict single-label output maintained.
    </td>
  </tr>
</table>

#### 📌 Summary

The robustness test highlights how prompt design behaves under realistic, multi-sentence customer feedback.

In V1:

- Positive classification remained relatively stable (93.33%) due to clear sentiment signals.
- Negative and neutral inputs showed severe instability (40% and 53.33%).
- Significant taxonomy drift occurred (e.g., “Product Complaint / Dissatisfaction”).
- Outputs included explanatory sentences and invented category labels.
- Not automation-safe under complex input.

In V3:

- All categories achieved 100% consistency.
- No label expansion or explanatory text appeared.
- The fixed taxonomy and few-shot reinforcement prevented drift even with nuanced inputs.

Key Insight:

Zero-shot prompts may appear stable on simple inputs but degrade under realistic complexity.  
Few-shot prompting combined with strict output constraints ensures robustness, taxonomy control, and production-level reliability.


## STEP 5.2 Robustness Testing – Data Extraction (Extended Input)

The original customer feedback used in Versions 1 and 2 was relatively simple:

"I ordered item #12345 on March 15th. The delivery was fast but the packaging was damaged."

While sufficient for baseline evaluation, this input did not fully test the reasoning capabilities of the extraction prompts.

To evaluate robustness under increased complexity, a more detailed customer feedback example was introduced:

"I placed order #78910 on April 2nd. The package arrived two days later, which was faster than expected. However, the outer box was torn, and one corner of the product packaging was slightly crushed. The product itself works fine."

This extended input includes:

- A different order identifier
- A new order date
- Implicit delivery speed information
- Multiple packaging condition details
- Additional contextual information unrelated to extraction fields

The goal of this robustness test was to:

- Evaluate reasoning stability under multi-signal input
- Assess schema adherence under increased textual complexity
- Compare extraction behavior between Version 1 (unstructured) and Version 3 (Chain-of-Thought + JSON schema)
- Confirm that structured prompts maintain machine-readable output even with richer input

Execution reused the existing evaluation functions and appended results to the existing result files to maintain experimental continuity.

This robustness phase validates that prompt improvements remain reliable beyond simplified baseline examples.


### STEP 5.2.1 - Complex Extraction Feedback (Robustness Test) ---

In [ ]:
# --- STEP 5.2.1 - Complex Extraction Feedback (Robustness Test) ---

complex_extraction_feedback = """
I placed order #78910 on April 2nd. The package arrived two days later,
which was faster than expected. However, the outer box was torn,
and one corner of the product packaging was slightly crushed.
The product itself works fine.
"""


### STEP 5.2.2 - Extraction v1 – Robustness Prompt (Unstructured) ---

In [ ]:
# --- STEP 5.2.2 - Extraction v1 – Robustness Prompt (Unstructured) ---

extraction_v1_complex_prompt = f"""
Extract information from this customer feedback:

"{complex_extraction_feedback}"
"""


### STEP 5.2.3 - Extraction v3 – Robustness Prompt (CoT + JSON) ---

In [ ]:
# --- STEP 5.2.3 - Extraction v3 – Robustness Prompt (CoT + JSON) ---

extraction_v3_complex_prompt = f"""
You are an information extraction system.

Extract structured data from the customer feedback below.

Customer Feedback:
"{complex_extraction_feedback}"

First, think step by step about:
- Order ID
- Order date
- Delivery speed
- Packaging condition

Then return ONLY the final answer in valid JSON format with EXACTLY these fields:

{{
  "order_id": "",
  "order_date": "",
  "delivery_speed": "",
  "packaging_condition": ""
}}

Rules:
- Do not include your reasoning in the output.
- Return only valid JSON.
- Use double quotes.
- If information is missing, use null.
"""


### STEP 5.2.4 - Extraction v1 – Robustness Test (Append) ---

In [ ]:
# --- STEP 5.2.4 - Extraction v1 – Robustness Test (Append) ---

run_unstructured_extraction_test(
    prompt=extraction_v1_complex_prompt,
    runs=15,
    log_file="extraction_v1_results.txt",
    label="Extraction v1 – Robustness (Complex Input)"
)


### STEP 5.2.5 - Extraction v3 – Robustness Test (Append) ---

In [ ]:
# --- STEP 5.2.5 - Extraction v3 – Robustness Test (Append) ---

run_extraction_test(
    prompt=extraction_v3_complex_prompt,
    runs=15,
    log_file="extraction_v3_results.txt",
    label="Extraction v3 – Robustness (Complex Input)"
)


### STEP 5.2.6 – Data Extraction – Robustness Evaluation (Extended Input)

#### 📊 Robustness Comparison Table (HTML)

<table border="1" cellpadding="8" cellspacing="0">
  <tr>
    <th>Version</th>
    <th>Exact-Match Consistency</th>
    <th>Structural Consistency</th>
    <th>Schema Stability</th>
    <th>Notes</th>
  </tr>
  <tr>
    <td>V1 – Zero-Shot</td>
    <td>40.00%</td>
    <td>0%</td>
    <td>Unstable</td>
    <td>
      Field expansion beyond required schema;
      inconsistent field naming;
      inclusion of extra fields (Product Functionality);
      optional "#" in order number;
      explanatory text included.
    </td>
  </tr>
  <tr>
    <td>V3 – CoT + JSON</td>
    <td>100.00%</td>
    <td>100%</td>
    <td>Fully Stable</td>
    <td>
      Strict adherence to defined JSON schema;
      no extra fields;
      no formatting drift;
      consistent interpretation of implicit delivery speed.
    </td>
  </tr>
</table>

#### 📌 Summary

The robustness test with extended, multi-signal input clearly demonstrates the impact of structured prompting.

In V1:

- The model extracted correct information but expanded the schema.
- Additional fields were introduced (e.g., Product Functionality).
- Field names varied across runs.
- Formatting and structure were inconsistent.
- Structural consistency remained 0%.
- Not safe for downstream automation.

In V3:

- Chain-of-Thought reasoning supported accurate multi-signal interpretation.
- Strict JSON schema prevented field expansion.
- Output remained fully machine-readable.
- Exact-match and structural consistency both reached 100%.
- Implicit delivery speed (“faster than expected”) was consistently interpreted.

Key Insight:

Under complex input, unstructured prompts amplify schema drift and variability.  
Combining reasoning guidance (CoT) with strict JSON constraints ensures robustness, schema adherence, and production-level reliability even in realistic scenarios.


# STEP 6 - Final Validation – Version 3 (Complex Inputs)


## STEP 6.1 - FINAL RUN: Sentiment v3 – Complex Positive ---

In [ ]:
# --- STEP 6.1 - FINAL RUN: Sentiment v3 – Complex Positive ---

run_classification_test(
    prompt=sentiment_v3_positive_prompt,
    runs=15,
    log_file="sentiment_v3_results.txt",
    label="Sentiment v3 – FINAL (Complex Positive)"
)

# --- FINAL RUN: Sentiment v3 – Complex Negative ---

run_classification_test(
    prompt=sentiment_v3_negative_prompt,
    runs=15,
    log_file="sentiment_v3_results.txt",
    label="Sentiment v3 – FINAL (Complex Negative)"
)

# --- FINAL RUN: Sentiment v3 – Complex Neutral ---

run_classification_test(
    prompt=sentiment_v3_neutral_prompt,
    runs=15,
    log_file="sentiment_v3_results.txt",
    label="Sentiment v3 – FINAL (Complex Neutral)"
)


## STEP 6.2 - FINAL RUN: Extraction v3 – Complex Input ---

In [ ]:
# --- STEP 6.2 - FINAL RUN: Extraction v3 – Complex Input ---

run_extraction_test(
    prompt=extraction_v3_complex_prompt,
    runs=15,
    log_file="extraction_v3_results.txt",
    label="Extraction v3 – FINAL (Complex Input)"
)


### STEP 6.3 - FINAL RUN: Product v3 – Structured + Few-Shot (Original Spec) ---

In [ ]:
# --- STEP 6.3 - FINAL RUN: Product v3 – Structured + Few-Shot (Original Spec) ---

run_generation_test(
    prompt=product_prompt_v3,
    runs=15,
    log_file="product_v3_results.txt",
    label="Product v3 – FINAL"
)

### STEP 6.4 – Final Validation – Consolidated Results

---

## 📊 Final Test Results (V3 Only – Complex Inputs)

<table border="1" cellpadding="8" cellspacing="0">
  <tr>
    <th>Task</th>
    <th>Version</th>
    <th>N Runs</th>
    <th>Exact-Match Consistency</th>
    <th>Structural Consistency</th>
    <th>Notes</th>
  </tr>
  <tr>
    <td>Sentiment Classification</td>
    <td>V3 – Few-Shot + Constraints</td>
    <td>15</td>
    <td>100%</td>
    <td>100%</td>
    <td>
      Fully deterministic single-label output.
      No taxonomy drift under complex input.
    </td>
  </tr>
  <tr>
    <td>Data Extraction</td>
    <td>V3 – CoT + JSON</td>
    <td>15</td>
    <td>100%</td>
    <td>100%</td>
    <td>
      Strict JSON schema maintained.
      Correct interpretation of implicit delivery speed.
      Machine-readable output preserved.
    </td>
  </tr>
  <tr>
    <td>Product Description</td>
    <td>V3 – Few-Shot + Structured</td>
    <td>15</td>
    <td>6.67%</td>
    <td>100%</td>
    <td>
      Fully stable structure (Headline / Description / Key Features).
      High creative variability in content.
    </td>
  </tr>
</table>

---

## 📊 Complete Lab Results Overview (All Versions & Tests)

<table border="1" cellpadding="8" cellspacing="0">
  <tr>
    <th>Task</th>
    <th>Version</th>
    <th>Test Type</th>
    <th>Exact-Match Consistency</th>
    <th>Structural Consistency</th>
  </tr>

  <!-- SENTIMENT -->
  <tr>
    <td>Sentiment</td>
    <td>V1 – Zero-Shot</td>
    <td>Baseline (15 runs)</td>
    <td>53.33%</td>
    <td>0%</td>
  </tr>
  <tr>
    <td>Sentiment</td>
    <td>V2 – Structured</td>
    <td>Baseline (15 runs)</td>
    <td>100%</td>
    <td>100%</td>
  </tr>
  <tr>
    <td>Sentiment</td>
    <td>V3 – Few-Shot</td>
    <td>Baseline (15 runs)</td>
    <td>100%</td>
    <td>100%</td>
  </tr>
  <tr>
    <td>Sentiment</td>
    <td>V1 – Zero-Shot</td>
    <td>Robustness</td>
    <td>
      Pos: 93.33%<br>
      Neg: 40%<br>
      Neu: 53.33%
    </td>
    <td>0%</td>
  </tr>
  <tr>
    <td>Sentiment</td>
    <td>V3 – Few-Shot</td>
    <td>Robustness</td>
    <td>100% (All)</td>
    <td>100%</td>
  </tr>

  <!-- EXTRACTION -->
  <tr>
    <td>Extraction</td>
    <td>V1 – Zero-Shot</td>
    <td>Baseline (15 runs)</td>
    <td>40%</td>
    <td>0%</td>
  </tr>
  <tr>
    <td>Extraction</td>
    <td>V2 – JSON</td>
    <td>Baseline (15 runs)</td>
    <td>100%</td>
    <td>100%</td>
  </tr>
  <tr>
    <td>Extraction</td>
    <td>V3 – CoT + JSON</td>
    <td>Baseline (15 runs)</td>
    <td>100%</td>
    <td>100%</td>
  </tr>
  <tr>
    <td>Extraction</td>
    <td>V1 – Zero-Shot</td>
    <td>Robustness</td>
    <td>40%</td>
    <td>0%</td>
  </tr>
  <tr>
    <td>Extraction</td>
    <td>V3 – CoT + JSON</td>
    <td>Robustness</td>
    <td>100%</td>
    <td>100%</td>
  </tr>

  <!-- PRODUCT -->
  <tr>
    <td>Product Description</td>
    <td>V1 – Zero-Shot</td>
    <td>Baseline (15 runs)</td>
    <td>40%</td>
    <td>0%</td>
  </tr>
  <tr>
    <td>Product Description</td>
    <td>V2 – Structured</td>
    <td>Baseline (15 runs)</td>
    <td>40%</td>
    <td>100%</td>
  </tr>
  <tr>
    <td>Product Description</td>
    <td>V3 – Few-Shot + Structured</td>
    <td>Baseline (15 runs)</td>
    <td>26.67%</td>
    <td>100%</td>
  </tr>
  <tr>
    <td>Product Description</td>
    <td>V3 – Few-Shot + Structured</td>
    <td>Final Test</td>
    <td>6.67%</td>
    <td>100%</td>
  </tr>

</table>

---

## 📌 Final Summary

This lab demonstrates three key insights about prompt engineering:

### 1️⃣ Classification Tasks
Strict label constraints + few-shot examples eliminate taxonomy drift and achieve full determinism (100% consistency), even under complex inputs.

### 2️⃣ Data Extraction Tasks
Explicit JSON schema definition is the dominant stabilizing factor.  
Combining schema constraints with reasoning guidance (CoT) ensures both robustness and machine-readability under multi-signal input.

### 3️⃣ Generative Tasks
Structural constraints stabilize formatting (100% structural consistency), but content variability remains inherent.  
Few-shot prompting improves stylistic alignment but does not enforce deterministic content output.

---

### 🎯 Overall Lab Outcome

- Clear measurable improvement from V1 → V3
- >80% consistency achieved for classification and extraction
- Structural determinism achieved where required
- Robustness validated under realistic complex inputs

The final prompts for sentiment and extraction are fully production-ready.  
The product description prompt is structurally production-safe but intentionally retains creative variability.


# STEP 7 - Further Prompt Improvements – Final Optimization Analysi

This section summarizes how each task can be further improved to increase determinism, robustness, and production safety.

---

## 1️⃣ Sentiment Classification – Further Improvements

### ✅ Current Strengths (V3)

- Strict label set  
- Single-word output  
- Few-shot examples  
- No explanation allowed  

### ⚠️ Remaining Weaknesses

- Does not explicitly forbid lowercase outputs  
- Does not explicitly forbid punctuation  
- No safeguard against subtle output drift  

---

### 🔧 Improved Version

```python
sentiment_prompt_v3_improved = '''
You are a sentiment classification system.

Classify the customer message into EXACTLY one of the following labels:

Positive
Negative
Neutral

Rules:
- Output must be EXACTLY one of the three labels above.
- Do not include punctuation.
- Do not include explanation.
- Do not include additional text.
- Output must match capitalization exactly.
- If uncertain, choose the closest valid label.

Customer Message:
"{feedback}"

Output:
'''
```


### 🎯 Why This Is Better

- Prevents lowercase drift
- Prevents punctuation leakage
- Eliminates formatting variation
- Increases production reliability

For classification tasks, strict output control is the primary stabilizer.

## 2️⃣ Product Description – Further Improvements

### ✅ Current Strengths (V3)

- Clear structure (Headline / Description / Key Features)
- Few-shot alignment
- Fixed bullet count

### ⚠️ Remaining Weaknesses

- Headline length not strictly constrained
- Sentence count not strictly enforced
- Minor formatting drift still possible

### 🔧 Improved Version

```python
product_prompt_v3_improved = '''
You are a professional e-commerce copywriter.

Follow this EXACT structure:

Headline:
- Maximum 8 words.

Description:
- Exactly 2 sentences.

Key Features:
- Exactly 3 bullet points.
- Each bullet must start with "- ".
- No extra blank lines.
- No additional commentary.

Now create a product description for:

Product: Wireless Mouse
Price: $29.99
'''
```


### 🎯 Why This Is Better

- Controls verbosity
- Enforces sentence count
- Reduces structural entropy
- Improves formatting determinism

Note: Increasing determinism reduces creativity. Marketing tasks require balance.


## 3️⃣ Data Extraction – Further Improvements

### ✅ Current Strengths (V3)

- Strict JSON schema
- Hidden Chain-of-Thought
- No reasoning leakage
- 100% structural consistency

### ⚠️ Remaining Weaknesses

- Does not explicitly forbid trailing text
- Does not forbid markdown code blocks
- No normalization guidance for phrasing

### 🔧 Improved Version
```python
extraction_prompt_v3_improved = '''
You are a structured information extraction system.

Extract data from the customer feedback below.

Customer Feedback:
"{feedback}"

Think step by step about:
- Order ID
- Order date
- Delivery speed
- Packaging condition

Then return ONLY valid JSON.

Rules:
- Output must be valid JSON only.
- Do not include backticks.
- Do not include explanations.
- Do not include additional text before or after JSON.
- Use double quotes.
- If information is missing, use null.
- Use short phrases (not full sentences).

JSON schema:

{
  "order_id": "",
  "order_date": "",
  "delivery_speed": "",
  "packaging_condition": ""
}
'''
```


### 🎯 Why This Is Better

- Prevents JSON wrapper drift
- Prevents markdown leakage
- Improves phrase consistency
- Preserves machine-readability under complexity

For extraction tasks, schema enforcement is the dominant reliability lever.

## 🔥 Core Insight From This Lab

Prompt improvement is not about:
- Adding more words
- Adding more examples
- Making prompts longer

It is about:
- Reducing degrees of freedom
- Constraining the output surface
- Matching constraints to task type
- Controlling formatting entropy

## 🧠 Final Takeaway - Improvement Levers by Task

### 📊 Prompt Stabilization Summary (HTML)

<table border="1" cellpadding="8" cellspacing="0">
  <tr>
    <th>Task</th>
    <th>Primary Stabilizer</th>
    <th>What Actually Improves It</th>
    <th>Risk If Not Applied</th>
  </tr>
  <tr>
    <td>Sentiment Classification</td>
    <td>Strict label enforcement</td>
    <td>
      Single-word constraint;<br>
      Exact label matching;<br>
      Capitalization control;<br>
      No explanation rule
    </td>
    <td>
      Taxonomy drift;<br>
      Label expansion;<br>
      Formatting inconsistency
    </td>
  </tr>
  <tr>
    <td>Product Description</td>
    <td>Explicit structural constraints</td>
    <td>
      Fixed section layout;<br>
      Sentence count control;<br>
      Bullet count enforcement;<br>
      Headline length limit
    </td>
    <td>
      Formatting entropy;<br>
      Verbosity drift;<br>
      Structural inconsistency
    </td>
  </tr>
  <tr>
    <td>Data Extraction</td>
    <td>JSON schema + output isolation</td>
    <td>
      Strict schema definition;<br>
      No extra fields;<br>
      No trailing text;<br>
      Machine-readable output only
    </td>
    <td>
      Schema drift;<br>
      Field expansion;<br>
      Parsing failure
    </td>
  </tr>
</table>

---

## 🎯 Engineering Lesson

The biggest improvements came from:

- Reducing degrees of freedom  
- Constraining output surface  
- Matching constraint type to task type  
- Controlling formatting entropy  

Prompt engineering is less about verbosity — and more about **controlled output design**.

The strongest improvements came from constraint engineering, not verbosity.


# STEP 8: Reflection

Throughout the lab, the most consistent failure patterns were format drift, label expansion, and schema instability. In zero-shot prompts, the model often produced semantically correct answers but varied the surface form — changing wording, adding explanations, inventing new category labels, or altering field names. For generative tasks, variability appeared in phrasing and structure, while extraction tasks showed schema drift and additional unexpected fields. These issues became more visible as the number of runs increased, proving that prompts that work once may fail under scale.

The techniques with the biggest impact were structural constraints and schema enforcement. For classification, strict label control combined with few-shot examples eliminated taxonomy drift and achieved full determinism. For extraction, explicit JSON schema definition was the dominant stabilizer — more impactful than Chain-of-Thought alone. Few-shot prompting improved stylistic alignment in generative tasks but did not significantly increase exact-match determinism. Overall, constraining the output surface had a greater effect than simply adding more examples.

I would also define evaluation metrics before running tests. I would clearly decide what I am measuring: Is the answer correct in meaning? Is the format correct? What level of variation is acceptable? What counts as a failure? Separating meaning from structure helps measure improvement more clearly. In real systems, correct format is often just as important as correct meaning.

Next time, I would test robustness earlier. I would use complex inputs, edge cases, ambiguous language, and real customer data if possible. This would help detect problems sooner and avoid optimizing prompts only for simple examples.

Finally, I would automate more of the testing process. Customer inputs would be stored in a file, prompts versioned separately, and evaluation scripts would run consistency tests automatically. This would make the process more efficient and closer to real AI development.


## Final Consistency Comparison

<table>
  <thead>
    <tr>
      <th>Task</th>
      <th>Version</th>
      <th>Exact-Match Consistency</th>
      <th>Structural Consistency</th>
      <th>Notes</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>Sentiment</td>
      <td>v1</td>
      <td>100%</td>
      <td>N/A</td>
      <td>Taxonomy drift observed</td>
    </tr>
    <tr>
      <td>Sentiment</td>
      <td>v3</td>
      <td>100%</td>
      <td>N/A</td>
      <td>Strict label enforcement</td>
    </tr>
    <tr>
      <td>Product</td>
      <td>v1</td>
      <td>6.67%</td>
      <td>N/A</td>
      <td>Format drift</td>
    </tr>
    <tr>
      <td>Product</td>
      <td>v2</td>
      <td>26.67%</td>
      <td>100%</td>
      <td>Best determinism</td>
    </tr>
    <tr>
      <td>Product</td>
      <td>v3</td>
      <td>6–13%</td>
      <td>100%</td>
      <td>Improved tone consistency</td>
    </tr>
    <tr>
      <td>Extraction</td>
      <td>v1</td>
      <td>73.33%</td>
      <td>0%</td>
      <td>Unstructured output</td>
    </tr>
    <tr>
      <td>Extraction</td>
      <td>v3</td>
      <td>100%</td>
      <td>100%</td>
      <td>Schema enforced</td>
    </tr>
  </tbody>
</table>


## Step 12 – What Works and What Needs Adjustment

This section evaluates how different prompt designs performed across task variations.

---

### 1️⃣ Sentiment Analysis

Best Version: Few-Shot + Structured Constraints (V3)

What Worked:
- Strict label enforcement prevented taxonomy drift.
- Single-word output ensured downstream compatibility.
- Few-shot examples reinforced classification boundaries.
- 100% consistency across extended test cases.

What Needs Adjustment:
- Few-shot examples can influence interpretation boundaries (e.g., shifting ambiguous classification from Negative to Neutral).
- For highly domain-specific sentiment tasks, more domain-aligned examples may be required.

Conclusion:
Structured constraints are essential. Few-shot improves semantic alignment rather than numerical consistency.

---

### 2️⃣ Product Description

Best Version: Structured Template (V2) and Few-Shot + Structured (V3)

What Worked:
- Explicit structure guarantees 100% structural consistency.
- Bullet count enforcement is highly reliable.
- Few-shot examples improved stylistic coherence and professional tone.

What Needs Adjustment:
- Exact-match consistency remains low across all versions.
- Ultra-strict constraints did not increase determinism.
- Role-based prompts improved tone but caused structural drift.

Key Insight:
Creative generation tasks cannot rely on exact-match metrics.
Structure enforcement is more important than literal wording consistency.

Conclusion:
For production systems, structural constraints are mandatory.
Few-shot improves tone but does not increase determinism.

---

### 3️⃣ Data Extraction

Best Version: Hidden Chain-of-Thought + JSON Schema (V3)

What Worked:
- JSON schema enforcement guaranteed structural stability.
- Hidden Chain-of-Thought improved reasoning for complex inputs.
- 100% structural consistency achieved in controlled tests.

What Needs Adjustment:
- Visible Chain-of-Thought severely reduced structural reliability.
- For simple inputs, CoT may not be necessary.
- Overengineering simple tasks adds complexity without benefit.

Key Insight:
CoT should remain hidden in production environments.
Schema constraints alone are sufficient for low-complexity extraction tasks.

Conclusion:
Extraction tasks benefit most from structured output enforcement.
CoT adds value primarily when reasoning complexity increases.

---

## Overall Findings

- Structure constraints are the most powerful consistency tool.
- Few-shot prompting improves semantic alignment and stylistic coherence.
- Chain-of-Thought improves reasoning stability but must remain hidden.
- Role-based prompts increase creativity but reduce format control.
- Determinism in creative tasks cannot be forced through stricter wording rules.

Final Lesson:
Prompt design must be adapted to task type.
There is no universal "best" prompt strategy.


## Reflection

Through systematic testing across 5, 10, and 15 iterations, I observed that consistency alone is not a sufficient indicator of prompt quality. In the Sentiment task, the baseline prompt achieved 100% consistency but failed to enforce the required taxonomy, inventing new labels. Structured constraints and few-shot prompting did not increase numerical consistency, but they significantly improved semantic reliability and production readiness by eliminating category drift and enforcing strict output formats.

For the Product Description task, the most impactful improvement came from structural constraints rather than few-shot prompting. Version 2 (structured template) increased exact-match consistency and achieved 100% structural stability. Few-shot prompting preserved structural consistency but did not improve determinism, highlighting that creative generation tasks cannot be forced into identical outputs through stricter wording rules. This reinforced the insight that structural consistency is a more meaningful metric than exact-match similarity for generative tasks.

In the Data Extraction task, schema enforcement proved to be the most powerful reliability mechanism. Chain-of-Thought reasoning improved stability under complex inputs, achieving 100% exact-match and structural consistency when combined with a strict JSON schema. However, allowing visible reasoning drastically reduced structural reliability. The key lesson from this lab is that prompt design must align with task type: classification requires label control, generation requires structural constraints, and extraction requires schema enforcement. There is no universal optimal strategy — prompt engineering is task-dependent and empirical.


For generative tasks, exact-match consistency is not an ideal metric. 
Structural consistency and constraint compliance are stronger indicators 
of production reliability.

Creative variation is desirable, provided the format and requirements 
are respected.


## Consistency Metrics by Task Type

Different task types require different consistency metrics. 
Using the same evaluation method across classification, generation, and extraction 
can lead to misleading conclusions.

Below is a breakdown of appropriate consistency metrics for each task type.

---

## 1️⃣ Sentiment Analysis (Classification Task)

### Primary Metric: Exact-Match Consistency

Definition:
The percentage of runs that produce the same classification label.

Formula:
(matching outputs ÷ total runs) × 100

Why it works:
- Classification tasks require deterministic category output.
- Outputs are short and categorical (Positive / Negative / Neutral).
- Identical labels are expected in production systems.

Risks Identified:
- Taxonomy drift (e.g., "Constructive Complaint")
- Extra explanatory text
- Label invention

Best Practice:
- Enforce strict allowed labels.
- Require single-word output.
- Use few-shot examples to reinforce decision boundaries.

Conclusion:
Exact-match consistency is the correct primary metric for classification tasks.

---

## 2️⃣ Product Description (Generative Task)

### Primary Metric: Structural Consistency

Definition:
Percentage of runs that maintain required format elements:
- Headline present
- Description present
- Exactly 3 bullet points
- No extra sections

Why it works:
- Creative wording variation is expected and desirable.
- Format stability matters more than identical phrasing.
- Production systems require predictable layout.

### Secondary Metrics:
- Word count compliance
- Sentence count compliance
- Tone alignment
- Bullet formatting consistency

Why Exact-Match Is Weak:
- Creative generation encourages lexical variation.
- Identical output is not a realistic production goal.
- High exact-match may indicate over-constrained or robotic text.

Conclusion:
For generative tasks, structural consistency and constraint compliance 
are stronger indicators of reliability than exact textual similarity.

---

## 3️⃣ Data Extraction (Structured Output Task)

### Primary Metric: Schema (Structural) Consistency

Definition:
Percentage of runs that:
- Return valid JSON
- Include all required fields
- Include no extra fields
- Use correct key names

Why it works:
- Extraction tasks must be machine-readable.
- Schema stability is critical for downstream automation.
- Format errors break pipelines.

### Secondary Metric: Exact-Match Consistency

Definition:
Percentage of runs producing identical JSON output.

Why it matters:
- Measures determinism under complex reasoning.
- Important for auditability and system stability.

Chain-of-Thought Consideration:
- Hidden CoT improves reasoning stability.
- Visible CoT reduces structural reliability.
- CoT should remain hidden in production.

Conclusion:
For extraction tasks, schema enforcement is the most important metric.
Exact-match consistency becomes relevant when reasoning complexity increases.

---

## Overall Insight

There is no universal consistency metric.

- Classification → Exact-match consistency
- Generation → Structural consistency
- Extraction → Schema consistency (+ determinism under complexity)

Prompt evaluation must align with task type and system requirements.
